## Анализ отзывов

В рамках данного кейса предлагается с помощью ИИ и машинного обучения поизучать отзывы людей на железнодорожные вокзалы Москвы. В рамках кейса предоставляются отзывы пользователей с Яндекс-карт, но возможно дополнительно использовать доступные в Интернет данные. 

Что ожидается в качестве результата:
* Инфографика, наглядно демонстрирующая полученные командой результаты
* Доклад, представляющий эти результаты
* Код в Jupyter Notebook с анализом данных

В данном ноутбуке показывается, как получить доступ к датасету, и первые шаги по его обработке с помощью доступных открытых моделей с HuggingFace, и с помощью YandexGPT. Можно использовать данный ноутбук как оправную точку для выполнения задания, расширив его дополнительными идеями, например:
* Кластеризация отзывов
* Обучение своих моделей для классификации тематики отзывов
* Использование предобученных NER-моделей
* Подробный анализ отзывов с помощью LLM

> В начале работы мы будем использовать модель с HuggingFace, поэтому работа будет быстрее, если вы будете запускать код на узле с поддержкой GPU

Для начала установим библиотеки:

In [ ]:
%pip install transformers openai


### Получение датасета

In [ ]:
!wget https://storage.yandexcloud.net/mypub/data/rail_reviews.zip

In [ ]:
!unzip *.zip

### Преобразуем данные в DataFrame

Как видите, каждый вокзал представлен своим набором отзывов следующего вида:

In [ ]:
import json
j = json.load(open('reviews_Belorussky_railway_station.json'))
j[0]

Для удобства, сведём все отзывы в одну табличку. Для этого нам нужно будет добавить в каждое JSON-описание отзыва название вокзала.

In [ ]:
import pandas as pd
import os
from glob import glob

res = []
for fn in glob('*.json'):
    j = json.load(open(fn))
    for x in j:
        x['station'] = fn.split('_')[1]
    res.extend(j)
    
df = pd.DataFrame(res)
df

Посмотрим на количество отзывов по вокзалам:

In [ ]:
df.groupby('station')['review_text'].count().plot.bar()

### Обогащение данных с помощью модели с HuggingFace

Для обогащения данных можно использовать различные нейросетевые модели. Для начала, попробуем определить тональность текста.

Попробуем использовать модель [blanchefort/rubert-base-cased-sentiment-rurewiews](https://huggingface.co/blanchefort/rubert-base-cased-sentiment-rurewiews) с HuggingFace, натренированную на отзывах, с длиной 512 токенов. По умолчанию код из карточки модели выдает следующие классы:
* 0: NEUTRAL
* 1: POSITIVE
* 2: NEGATIVE

> Код не учитывает возможность работы на GPU, поэтому придётся внести в код несколько исправлений, чтобы можно было быстрее выполнять код на GPU.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import BertTokenizerFast

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BertTokenizerFast.from_pretrained('blanchefort/rubert-base-cased-sentiment-rurewiews')
model = AutoModelForSequenceClassification.from_pretrained('blanchefort/rubert-base-cased-sentiment-rurewiews', return_dict=True)
model.to(device)

@torch.no_grad()
def predict(text):
    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')
    inputs = inputs.to(device)
    outputs = model(**inputs)
    predicted = torch.nn.functional.softmax(outputs.logits, dim=1)
    predicted = torch.argmax(predicted, dim=1).cpu().numpy()
    return predicted


Проверим, работает ли эта функция:

In [ ]:
predict([
    'Ничего так вокзал, обычный...',
    'Обожаю этот вокзал, там такие симпатичные бомжи!',
    'Ненавижу это вокзал, поезда всегда опаздывают минимум на 20 минут'])

Чтобы применить модель ко всему датасету, разобьем его на небольшие части, примерное по 200 отзывов, и предскажем тональность каждой из этих частей.

In [ ]:
from tqdm.auto import tqdm
import numpy as np 

res = []
for d in tqdm(np.array_split(df,10)):
    z = list(d['review_text'])
    p = predict(z)
    res.extend(p)

Для удобства вставим колонку `sentiment` в датасет, но при этом изменим кодирование: будем представлять негативный отзыв цифрой -1.

In [ ]:
df['sentiment'] = pd.Series(res).apply(lambda x: -1 if x==2 else x)

Запишем получившийся датасет на диск:

In [ ]:
df.to_csv('dataset_with_sentiment.csv',index=False)

Теперь можем посмотреть, какие вокзалы самые негативные или позитивные в Москве:

In [ ]:
def count_pos(x):
    return x[x==1].count()
def count_neg(x):
    return x[x==-1].count()


df.groupby('station').agg({'sentiment' : ['mean', count_pos, count_neg ], 'review_text': 'count'})

В этом месте вы можете остановить виртуальную машину с GPU и перейти на более дешевый вариант **c1.4**

### Используем YandexGPT для извлечения смысла из отзывов

Попробуем использовать большую языковую модель для извлечения структурированной информации из текстов отзывов. Обратимся к YandexGPT 5.1 через [OpenAI-compatible Responses API](https://aistudio.yandex.ru/docs/en/ai-studio/) Yandex AI Studio.


Для работы с YandexGPT, нам потребуются значения `api_key` и `folder_id`. Их надо взять из секретов датасферы.

In [ ]:
import os

folder_id = os.environ['folder_id']
api_key = os.environ['api_key']
print(f"Using folder {folder_id}")

Создадим функцию для вызова модели Yandex GPT:

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)
model = f"gpt://{folder_id}/yandexgpt-5.1"

def gpt(messages,
        system_message=None):
    if isinstance(messages,str):
        messages = [{ "role" : "user", "content" : messages }]
        if system_message is not None:
            messages.insert(0, { "role" : "system", "content" : system_message })
    response = client.responses.create(model=model, input=messages)
    return response.output_text

gpt("Расскажи анекдот про JSON и XML")


Для обработки, чтобы ускорить процесс демонстрации, выберем по 10 записей из каждого вокзала:

In [ ]:
import pandas as pd

df = pd.read_csv('dataset_with_sentiment.csv')
df_sample = df.groupby('station').apply(lambda x: x.sample(10)).reset_index(drop=True)
df_sample

Попробуем выделить смысл из текста отзыва. Для примера выберем три аспекта отзывов: транспортная доступность вокзала, его чистота и точность хождения поездов. По каждому из этих аспектов попробуем выделить оценку, также в целом положительные и отрицательные моменты, и список тегов, о чем этот отзыв.

In [ ]:
prompt = """
Прочитай следующий отзыв о вокзале в тройных обратных кавычках: ````{}```.
Из этого отзыва тебе необходимо выделить следующую информацию:
* sentiment - этот отзыв позитивный (positive), негативный (negative) или нейтральный (neutral)
* transport - транспортная доступность вокзала, по шкале 1..5, если об это говорится в отзыве. Если нет - 0
* cleanliness - чистота вокзала, по шкале 1..5, если об это говорится в отзыве. Если нет - 0
* schedule - точность хождения поездов, по шкале 1..5, если об это говорится в отзыве. Если нет - 0
* positive - краткий список позитивных моментов из отзыва
* negative - краткий список негативных моментов из отзыва
* tags - краткий список тегов, о чем этот отзыв, например: cleanliness, transport, trains
Результат необходимо вернуть в формате JSON такого вида:
{{
  "sentiment" : "...",
  "transport" : ...,
  "cleanliness" : ...,
  "schedule" : ...,
  "positive" : ["...", ...],
  "negative" : ["...", ...],
  "tags" : ["..."]
}}
"""

txt = df['review_text'].iloc[0]
print(txt)
res = gpt(prompt.format(txt))
res


Чтобы гарантировать возврат ответа в формате JSON, используем так называемый structured output. Вместо того чтобы разбирать текст ответа, зададим модель данных на Python с помощью Pydantic: модель вернёт готовый объект с полями нужных типов.


In [ ]:
from pydantic import BaseModel

class Review(BaseModel):
    sentiment: str
    transport: int
    cleanliness: int
    schedule: int
    positive: list[str]
    negative: list[str]
    tags: list[str]

txt = df['review_text'].iloc[0]
print(txt)
review = client.responses.parse(
    model=model,
    input=prompt.format(txt),
    text_format=Review,
).output_parsed
review


Сохраним колонку `sentiment`, распознанную моделью HuggingFace.

In [ ]:
pmap = { -1 : 'negative', 0 : 'neutral', 1 : 'positive' }
df_sample['hf_sentiment'] = df_sample['sentiment'].apply(lambda x: pmap[x])
df_sample.drop(columns=['sentiment'],inplace=True)

Теперь пройдёмся по всем строкам таблицы и извлечём информацию с помощью LLM. Для начала добавим новые пустые поля в таблицу, для хранения извлечённых значений:

In [ ]:
import numpy as np

for f in ["sentiment","transport","cleanliness", "schedule", "positive", "negative", "tags"]:
    df_sample[f]=np.nan

Теперь собственно займёмся извлечением. Structured output возвращает сразу объект `Review` нужного типа, поэтому разбирать JSON вручную не нужно. Если отзыв уже распознан (поле `positive` заполнено), пропускаем его.


In [ ]:
from tqdm.auto import tqdm

for i,r in tqdm(df_sample.iterrows(),total=len(df_sample)):
    txt = r['review_text']
    if not pd.isnull(r['positive']):
        continue # уже распознано
    review = client.responses.parse(
        model=model,
        input=prompt.format(txt),
        text_format=Review,
    ).output_parsed
    df_sample.at[i,'sentiment'] = review.sentiment
    df_sample.at[i,'transport'] = review.transport
    df_sample.at[i,'cleanliness'] = review.cleanliness
    df_sample.at[i,'schedule'] = review.schedule
    df_sample.at[i,'positive'] = ','.join(review.positive)
    df_sample.at[i,'negative'] = ','.join(review.negative)
    df_sample.at[i,'tags'] = ','.join(review.tags)


Посмотрим на результат:

In [ ]:
df_sample.to_csv('dataset_sample_with_GPT.csv',index=False)

In [ ]:
df_sample = pd.read_csv('dataset_sample_with_GPT.csv')
df_sample

## Делаем выводы и строим инфографику 

Посмотрим, насколько совпадают предсказания sentiment между YandexGPT и моделью с HuggingFace:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(df_sample['hf_sentiment'],df_sample['sentiment'])

Посмотрим на средние значения показателей по всем вокзалам. Для этого сначала заменим нулевые значения на NaN, и затем осуществим аггрегацию:

In [ ]:
for f in ["transport","cleanliness", "schedule"]:
    df_sample[f] = df_sample[f].apply(lambda x: np.nan if x==0 else x)

df_sample.groupby('station').agg( {'transport' : 'mean', 'cleanliness' : 'mean', 'schedule' : 'mean'})

Такую же информацию можно представить в виде графика:

In [ ]:
df_sample.groupby('station').agg(
    {'transport' : 'mean', 
     'cleanliness' : 'mean', 
     'schedule' : 'mean'}).plot.bar()

Теперь научимся извлекать интересующие нас отзывы по тегам. Для этого посмотрим на список всех тегов:

In [ ]:
tags = list(df_sample['tags'].apply(lambda x: str(x).split(',')))
tags = set(sum(tags,[])) - { 'nan', '' }
tags

Опишем функцию `tag_lookup`, которая будет возвращать фрагмент таблицы, в который входит интересующий нас тег:

In [ ]:
def tag_lookup(tag):
    return df_sample[df_sample['tags'].apply(lambda x: tag in str(x))]

tag_lookup('food')[['station','review_text']]

Посмотрим, на каком вокзале есть музей:

In [ ]:
tag_lookup('museum')[['station','review_text']]

## Подведение итогов

Зададимся задачей подытожить все положительные и отрицательные моменты для каждого из вокзалов. Сначала объединим все значения колонок `positive` и `negative` для каждого из вокзалов:

In [ ]:
join = lambda x : ', '.join([t for t in x if isinstance(t,str) and len(t)>0])

df_stations = df_sample.groupby('station').agg({ 'positive' : join, 'negative' : join }).reset_index()
df_stations

Теперь применим большую языковую модель для того, чтобы суммаризировать все факты в один конкретный текст:

In [ ]:
prompt_sum = """
Пожалуйста, прочитай список отзывов о вокзале ниже в тройных обратных кавычках и запиши краткое содержание
всего прочитанного текста в виде нескольких абзацев текста. Отзывы: ```{}```
"""

def summarize(x):
    res = gpt(prompt_sum.format(x))
    return res

df_stations['pos_summary'] = df_stations['positive'].apply(summarize)
df_stations['neg_summary'] = df_stations['negative'].apply(summarize)

In [ ]:
from IPython.display import display
with pd.option_context('display.max_colwidth', 0):
    display(df_stations[['station','pos_summary','neg_summary']])

## Заключение

Yandex Cloud позволяет вам как использовать ресурсы с GPU для запуска готовых предобученных моделей, так и предоставляет готовые сервисы и фундаментальные модели, которые можно использовать для решения ваших задач. Хотя мы это не показали, ресурсы GPU можно использовать и для обучения моделей!